# 📊 Industry-Oriented Marketing Campaign Performance Analysis
> **Tools:** Python · Pandas · NumPy · Matplotlib · Seaborn · Scikit-learn

---
**Objective:** Measure campaign effectiveness, ROI, and customer behaviour to drive data-driven marketing decisions.

| Section | Description |
|---|---|
| 1 | Dataset generation & ingestion |
| 2 | Data cleaning & preprocessing |
| 3 | EDA – reach & engagement |
| 4 | Conversion & ROI analysis |
| 5 | Customer behaviour analysis |
| 6 | Multi-campaign comparison |
| 7 | ML – clustering & regression |
| 8 | Visualisations |
| 9 | Actionable insights |

## 0 · Imports & Configuration

In [ ]:
import warnings; warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.ticker import FuncFormatter
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import os, sys

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

PALETTE = ['#2563EB','#7C3AED','#059669','#DC2626',
           '#D97706','#0891B2','#BE185D','#65A30D']
sns.set_theme(style='whitegrid', palette=PALETTE, font_scale=1.05)

money = FuncFormatter(lambda x,_: f'${x:,.0f}')
pct   = FuncFormatter(lambda x,_: f'{x:.1f}%')

print('✅ All libraries imported successfully')

## 1 · Dataset Generation & Ingestion

In [ ]:
# Run this cell once to create the dataset
import subprocess, sys
result = subprocess.run([sys.executable, 'src/generate_data.py'], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('ERROR:', result.stderr)

In [ ]:
df = pd.read_csv('data/marketing_campaigns.csv', parse_dates=['date'])
print(f'Shape : {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

## 2 · Data Cleaning & Preprocessing

In [ ]:
print('Missing values:\n', df.isnull().sum()[df.isnull().sum() > 0])
print('\nDtypes:\n', df.dtypes)
print('\nDuplicates:', df.duplicated().sum())

In [ ]:
before = len(df)
df.drop_duplicates(inplace=True)
df.dropna(subset=['campaign_name','impressions','clicks','conversions','spend'], inplace=True)
df = df[(df['impressions'] > 0) & (df['spend'] > 0) & (df['clicks'] <= df['impressions'])]

# Derived metrics
df['ctr']              = (df['clicks']      / df['impressions'] * 100).round(2)
df['cvr']              = (df['conversions'] / df['clicks']      * 100).round(2)
df['roi']              = ((df['revenue'] - df['spend']) / df['spend'] * 100).round(2)
df['roas']             = (df['revenue'] / df['spend']).round(2)
df['cpa']              = (df['spend'] / df['conversions'].replace(0, np.nan)).round(2)
df['profit']           = df['revenue'] - df['spend']
df['budget_util_pct']  = (df['spend'] / df['budget_allocated'] * 100).round(2)
df['month_num']        = df['date'].dt.month

print(f'Rows after cleaning: {len(df):,}  (removed {before-len(df)})')
df[['ctr','cvr','roi','roas','cpa','profit','budget_util_pct']].describe().round(2)

## 3 · Campaign Reach & Engagement

In [ ]:
reach = df.groupby('campaign_name').agg(
    impressions=('impressions','sum'),
    clicks=('clicks','sum'),
    avg_ctr=('ctr','mean')
).sort_values('impressions', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Reach & Engagement', fontsize=14, fontweight='bold')

axes[0].barh(reach.index, reach['impressions']/1e6, color=PALETTE[:len(reach)])
axes[0].xaxis.set_major_formatter(FuncFormatter(lambda x,_: f'{x:.0f}M'))
axes[0].set_title('Total Impressions')

axes[1].barh(reach.index, reach['avg_ctr'], color=PALETTE[:len(reach)])
axes[1].xaxis.set_major_formatter(pct)
axes[1].set_title('Avg Click-Through Rate (CTR)')

plt.tight_layout(); plt.show()
reach

## 4 · Conversion & ROI Analysis

In [ ]:
camp_kpi = df.groupby('campaign_name').agg(
    spend=('spend','sum'), revenue=('revenue','sum'),
    conversions=('conversions','sum'),
    avg_roi=('roi','mean'), avg_roas=('roas','mean'),
    avg_cpa=('cpa','mean')
).reset_index()
camp_kpi['profit'] = camp_kpi['revenue'] - camp_kpi['spend']
camp_kpi.sort_values('avg_roi', ascending=False)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Conversion & ROI Metrics', fontsize=14, fontweight='bold')

sorted_roi = camp_kpi.sort_values('avg_roi')
colors = ['#DC2626' if v < 100 else '#059669' for v in sorted_roi['avg_roi']]
axes[0].barh(sorted_roi['campaign_name'], sorted_roi['avg_roi'], color=colors)
axes[0].xaxis.set_major_formatter(pct)
axes[0].axvline(0, color='black', linewidth=0.8)
axes[0].set_title('Average ROI')

axes[1].bar(camp_kpi['campaign_name'], camp_kpi['avg_roas'], color=PALETTE)
axes[1].set_xticklabels(camp_kpi['campaign_name'], rotation=30, ha='right', fontsize=8)
axes[1].set_title('Average ROAS')

axes[2].bar(camp_kpi['campaign_name'], camp_kpi['avg_cpa'], color=PALETTE)
axes[2].set_xticklabels(camp_kpi['campaign_name'], rotation=30, ha='right', fontsize=8)
axes[2].yaxis.set_major_formatter(money)
axes[2].set_title('Average CPA ($)')

plt.tight_layout(); plt.show()

## 5 · Customer Behaviour Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Customer Behaviour', fontsize=14, fontweight='bold')

seg = df.groupby('customer_segment')[['revenue','conversions']].sum().reset_index()
axes[0].bar(seg['customer_segment'], seg['revenue']/1e6, color=PALETTE[:4])
axes[0].set_xticklabels(seg['customer_segment'], rotation=15, ha='right', fontsize=8)
axes[0].yaxis.set_major_formatter(FuncFormatter(lambda x,_: f'${x:.1f}M'))
axes[0].set_title('Revenue by Segment')

dev = df.groupby('device_type')['revenue'].sum()
axes[1].pie(dev, labels=dev.index, autopct='%1.1f%%',
            colors=PALETTE[:3], startangle=140,
            wedgeprops={'edgecolor':'white','linewidth':2})
axes[1].set_title('Revenue by Device')

bounce = df.groupby('channel')['bounce_rate'].mean().sort_values()
axes[2].barh(bounce.index, bounce.values, color=PALETTE[:len(bounce)])
axes[2].xaxis.set_major_formatter(pct)
axes[2].set_title('Avg Bounce Rate by Channel')

plt.tight_layout(); plt.show()

## 6 · Multi-Campaign Comparison

In [ ]:
pivot_roas = df.pivot_table(values='roas', index='campaign_name',
                             columns='quarter', aggfunc='mean')
fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(pivot_roas, annot=True, fmt='.1f', cmap='RdYlGn',
            linewidths=0.5, ax=ax, cbar_kws={'label':'ROAS'})
ax.set_title('ROAS Heatmap — Campaign × Quarter', fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Monthly revenue trends per campaign
monthly = df.groupby(['month_num','campaign_name'])['revenue'].sum().reset_index()
fig, ax = plt.subplots(figsize=(14, 6))
for i, camp in enumerate(df['campaign_name'].unique()):
    sub = monthly[monthly['campaign_name'] == camp]
    ax.plot(sub['month_num'], sub['revenue']/1e3, marker='o',
            label=camp.replace('_',' '), color=PALETTE[i % len(PALETTE)])
ax.set_xticks(range(1,13))
ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun',
                    'Jul','Aug','Sep','Oct','Nov','Dec'])
ax.yaxis.set_major_formatter(FuncFormatter(lambda x,_: f'${x:.0f}K'))
ax.set_title('Monthly Revenue Trend by Campaign', fontweight='bold')
ax.legend(fontsize=8, bbox_to_anchor=(1.01,1), loc='upper left')
plt.tight_layout(); plt.show()

## 7 · ML — Clustering & Regression

In [ ]:
features = ['spend','revenue','ctr','cvr','roi','roas','cpa',
            'bounce_rate','avg_time_on_site_s','avg_pages_per_visit']
ml_df = df[features].copy().dropna()

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(ml_df)

# Elbow
inertias = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(7,4))
plt.plot(range(2,9), inertias, marker='o', color=PALETTE[0])
plt.axvline(4, linestyle='--', color=PALETTE[3], label='k=4')
plt.title('K-Means Elbow Curve'); plt.xlabel('k'); plt.ylabel('Inertia')
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
best_k = 4
km = KMeans(n_clusters=best_k, random_state=42, n_init=10)
ml_df['cluster'] = km.fit_predict(X_scaled)

pca  = PCA(n_components=2, random_state=42)
X_2d = pca.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for c in range(best_k):
    mask = ml_df['cluster'] == c
    axes[0].scatter(X_2d[mask,0], X_2d[mask,1], alpha=0.4, s=12,
                    color=PALETTE[c], label=f'Cluster {c}')
axes[0].set_title('Campaign Clusters (PCA 2D)')
axes[0].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
axes[0].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
axes[0].legend()

profile = ml_df.groupby('cluster')[['roi','roas','ctr','cvr']].mean()
profile_norm = (profile - profile.min()) / (profile.max() - profile.min())
profile_norm.T.plot(kind='bar', ax=axes[1], color=PALETTE[:best_k], edgecolor='white')
axes[1].set_title('Cluster Profiles (normalised)')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)
axes[1].legend(title='Cluster')

plt.tight_layout(); plt.show()
print('\nCluster Centres (mean metrics):')
ml_df.groupby('cluster')[['spend','revenue','roi','roas','cpa']].mean().round(2)

In [ ]:
# Linear regression: spend → revenue
lr = LinearRegression()
lr.fit(df[['spend']], df['revenue'])
r2 = r2_score(df['revenue'], lr.predict(df[['spend']]))

sample = df.sample(500, random_state=42)
plt.figure(figsize=(9,5))
plt.scatter(sample['spend'], sample['revenue'], alpha=0.25, s=15, color=PALETTE[0])
sp = np.linspace(df['spend'].min(), df['spend'].max(), 200)
plt.plot(sp, lr.predict(sp.reshape(-1,1)), color=PALETTE[3],
         linewidth=2, label=f'Fit  R²={r2:.3f}')
plt.gca().xaxis.set_major_formatter(money)
plt.gca().yaxis.set_major_formatter(money)
plt.title('Spend → Revenue Regression', fontweight='bold')
plt.xlabel('Spend ($)'); plt.ylabel('Revenue ($)')
plt.legend(); plt.tight_layout(); plt.show()
print(f'Coefficient: {lr.coef_[0]:.2f}  |  Intercept: {lr.intercept_:.2f}  |  R²: {r2:.4f}')

## 8 · Pre-Generated Visualisations

In [ ]:
from IPython.display import Image, display
for fig_file in sorted(os.listdir('outputs')):
    if fig_file.endswith('.png'):
        print(f'\n--- {fig_file} ---')
        display(Image(filename=f'outputs/{fig_file}', width=900))

## 9 · Actionable Insights & Recommendations

| # | Insight | Recommendation |
|---|---------|---------------|
| 1 | **Email Newsletter** has the best ROI & ROAS | Scale budget; personalise subject lines by segment |
| 2 | **Retargeting Display** drives the most revenue | Increase frequency caps & expand retargeting audiences |
| 3 | **LinkedIn B2B** has the lowest ROI | Audit creative, audience match & landing page; pause if no improvement |
| 4 | **YouTube Video** has the lowest CTR | Improve first-5-second hook; test shorter formats |
| 5 | Bounce rates are high for Video/Display | Align ad creative with landing page copy & CTA |
| 6 | Q4 revenue peaks | Pre-allocate 30–40% of budget to Q4 campaigns |
| 7 | Mobile drives most clicks but lower CVR | Optimise mobile checkout flow and page load speed |
| 8 | Millennials & Gen X have highest ROI | Focus top-performing campaigns on these cohorts |
| 9 | Cluster 0 profiles align with high-ROI traits | Use cluster 0 feature values as creative/targeting brief |
| 10 | Budget utilisation averages 81% | Prevent over-spending; reallocate unused budget dynamically |
